In [ ]:
import cv2
import os

In [ ]:
labels = []
def load_images_from_folder(folder):
    data = []
    for filename in os.listdir(folder):
        if filename.lower().endswith('.jpg'):
            img = cv2.imread(os.path.join(folder, filename))
            data.append(img)
            
            labels.append(filename)
    return data
dog_images = load_images_from_folder('training/dog')
cat_images = load_images_from_folder('training/cat')

In [ ]:
import numpy as np
dog_images = np.array(dog_images)
cat_images = np.array(cat_images)

print(dog_images.shape)
print(cat_images.shape)

print(dog_images.reshape(10, -1).shape)

print(labels)

(10, 224, 224, 3)
(10, 224, 224, 3)
(10, 150528)


In [ ]:
merged_features = np.concatenate((dog_images.reshape(10, -1), cat_images.reshape(10, -1)), axis = 0)
print(merged_features.shape)

(20, 150528)


In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=2).fit(merged_features)
print(kmeans.labels_)

[0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [ ]:
from transformers import CLIPProcessor, CLIPModel
import torch

clip_processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")

def magic_function(images):
    inputs = clip_processor(images = images, return_tensors = "pt", padding = True)
    with torch.no_grad():
        features = clip_model.get_image_features(**inputs)
    return features.numpy()
dog_images_features = magic_function(list(dog_images))
cat_images_features = magic_function(list(cat_images))

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

...

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

In [ ]:
merged_features = np.concatenate((dog_images_features, cat_images_features), axis = 0)
print(merged_features)

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=2, random_state = 0).fit(merged_features)
print(kmeans.labels_)

for real_label, predicted_label in zip(labels, kmeans.labels__):
    print(f'Cluster {predicted_label} : {real_label}')